In [2]:
import os
import re
import joblib
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import nltk
from nltk.corpus import stopwords


In [5]:
# Load the dataset
# In Google Colab, upload the final CSV as /content/Somali_News_Dataset.csv.
DATA_PATH_CANDIDATES = [
    '/content/Somali_News_Dataset.csv',
    '/content/Somali_News_Dataset_Last_version.csv',
    'Somali_News_Dataset.csv',
    'Somali_News_Dataset_Last_version.csv',
    '/mnt/data/Somali_News_Dataset_Last_version (1)(1).csv'
]

data_path = next((path for path in DATA_PATH_CANDIDATES if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError(
        'Dataset CSV not found. Upload it to Colab as /content/Somali_News_Dataset.csv '
        'or update DATA_PATH_CANDIDATES with the correct path.'
    )

df = pd.read_csv(data_path)
print(f'Dataset loaded from: {data_path}')


Dataset loaded from: /content/Somali_News_Dataset.csv


In [6]:
print(f"Dataset loaded. Shape: {df.shape}")
print("Sample data:")
print(df.head())

Dataset loaded. Shape: (7240, 3)
Sample data:
                                               title  \
0  War Deg-Deg Ah: Qarax Ka Dhacay Xero Damaanyo ...   
1  Soomaaliya iyo Ciraaq oo ka wada-hadlay xoojin...   
2  Taliyaha Guutada 26-aad ee CXD oo lagu dilay H...   
3  Madaxweyne Xasan Sheekh”Qaddiyadda Falastiin w...   
4  Taliye ku xigeenka Ciidanka Uganda oo Muqdisho...   

                                             content  category  
0  Qarax ayaa ka dhacay Xerada Damaanyo ee magaal...    Dagaal  
1  Wasiirka Gaashaandhigga Xukuumadda Federaalka ...  Siyaasad  
2  Warar rasmi ah ayaa xaqiijinaya in taliyihii G...    Dagaal  
3  Madaxweyne Xasan Sheekh”Qaddiyadda Falastiin w...  Siyaasad  
4  Taliyaha Ciidanka Dhulka Xoogga Dalka Soomaali...    Dagaal  


In [7]:
df['text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')

In [8]:
# 3. Custom text cleaner Transformer to use in pipeline
class TextCleaner(TransformerMixin):
    def transform(self, X, **transform_params):
        return [self.clean_text(text) for text in X]

    def fit(self, X, y=None, **fit_params):
        return self

    @staticmethod
    def clean_text(text):
        text = text.lower()  # Lowercase
        text = re.sub(r'\d+', '', text)  # Remove digits
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
        text = text.strip()
        return text


In [9]:
# 4. Prepare stopwords list.
# Note: the original experiment used English stopwords. Replace this with your Somali stopword list if available.
try:
    nltk.download('stopwords', quiet=True)
    stop_words = stopwords.words('english')
except Exception:
    stop_words = 'english'


In [10]:
# 5. Stratified 80/20 holdout split for reproducible evaluation
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

print('Training set size:', len(X_train))
print('Test set size:', len(X_test))
print('Training class distribution:')
print(y_train.value_counts().sort_index())
print('Test class distribution:')
print(y_test.value_counts().sort_index())


Training set size: 5792
Test set size: 1448
Training class distribution:
category
Caafimaad        724
Ciyaaro          724
Dagaal           724
Diini            724
Ganacsi          724
Madadaalo        724
Siyaasad         724
Tiknoolajiyad    724
Name: count, dtype: int64
Test class distribution:
category
Caafimaad        181
Ciyaaro          181
Dagaal           181
Diini            181
Ganacsi          181
Madadaalo        181
Siyaasad         181
Tiknoolajiyad    181
Name: count, dtype: int64


In [11]:
# 6. Proposed model: TF-IDF unigram-bigram features + Linear SVM
pipeline = Pipeline([
    ('cleaner', TextCleaner()),
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words=stop_words, ngram_range=(1, 2))),
    ('svm', LinearSVC(C=1.0, random_state=42))
])


In [12]:
# 7. Train the model
print("Training SVM classifier...")
pipeline.fit(X_train, y_train)

Training SVM classifier...


Pipeline(steps=[('cleaner', <__main__.TextCleaner object at 0x7ccf7ce49700>),
                ('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words=['a', 'about', 'above', 'after',
                                             'again', 'against', 'ain', 'all',
                                             'am', 'an', 'and', 'any', 'are',
                                             'aren', "aren't", 'as', 'at', 'be',
                                             'because', 'been', 'before',
                                             'being', 'below', 'between',
                                             'both', 'but', 'by', 'can',
                                             'couldn', "couldn't", ...])),
                ('svm', LinearSVC(random_state=42))])

In [13]:
# 8. Predict on test data
y_pred = pipeline.predict(X_test)


In [14]:
# 9. Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9358
Classification Report:
               precision    recall  f1-score   support

    Caafimaad       0.97      0.93      0.95       181
      Ciyaaro       0.99      0.93      0.96       181
       Dagaal       0.89      0.93      0.91       181
        Diini       1.00      0.98      0.99       181
      Ganacsi       0.94      0.90      0.92       181
    Madadaalo       0.99      0.98      0.98       181
     Siyaasad       0.79      0.89      0.84       181
Tiknoolajiyad       0.95      0.95      0.95       181

     accuracy                           0.94      1448
    macro avg       0.94      0.94      0.94      1448
 weighted avg       0.94      0.94      0.94      1448



## Supervised baseline comparison

The following cell compares the proposed Linear SVM model with supervised baselines requested by the reviewer: Multinomial Naive Bayes, Logistic Regression, Random Forest, and Linear SVM n-gram variants. All models use the same stratified 80/20 split, random seed, preprocessing step, and evaluation metrics. LDA and LSI should be reported only as exploratory unsupervised topic models, not as fair supervised baselines.


In [15]:
# 10. Supervised baseline comparison
# All models use the same X_train, X_test, y_train, y_test produced by the stratified split above.

baseline_configs = [
    {
        'model_name': 'Multinomial Naive Bayes',
        'feature_setting': 'TF-IDF unigram + bigram',
        'ngram_range': (1, 2),
        'classifier': MultinomialNB()
    },
    {
        'model_name': 'Logistic Regression',
        'feature_setting': 'TF-IDF unigram + bigram',
        'ngram_range': (1, 2),
        'classifier': LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1)
    },
    {
        'model_name': 'Random Forest',
        'feature_setting': 'TF-IDF unigram + bigram',
        'ngram_range': (1, 2),
        'classifier': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    },
    {
        'model_name': 'Linear SVM',
        'feature_setting': 'TF-IDF unigram',
        'ngram_range': (1, 1),
        'classifier': LinearSVC(C=1.0, random_state=42)
    },
    {
        'model_name': 'Linear SVM',
        'feature_setting': 'TF-IDF unigram + bigram [proposed]',
        'ngram_range': (1, 2),
        'classifier': LinearSVC(C=1.0, random_state=42)
    },
    {
        'model_name': 'Linear SVM',
        'feature_setting': 'TF-IDF unigram + bigram + trigram',
        'ngram_range': (1, 3),
        'classifier': LinearSVC(C=1.0, random_state=42)
    }
]

baseline_results = []
baseline_reports = {}

for config in baseline_configs:
    print(f"Training {config['model_name']} | {config['feature_setting']}...")
    model_pipeline = Pipeline([
        ('cleaner', TextCleaner()),
        ('tfidf', TfidfVectorizer(
            max_features=5000,
            stop_words=stop_words,
            ngram_range=config['ngram_range']
        )),
        ('classifier', config['classifier'])
    ])

    model_pipeline.fit(X_train, y_train)
    predictions = model_pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_test, predictions, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_test, predictions, average='weighted', zero_division=0
    )

    baseline_results.append({
        'Model': config['model_name'],
        'Feature Setting': config['feature_setting'],
        'Accuracy': round(accuracy, 4),
        'Macro Precision': round(macro_p, 4),
        'Macro Recall': round(macro_r, 4),
        'Macro F1': round(macro_f1, 4),
        'Weighted F1': round(weighted_f1, 4)
    })

    baseline_reports[f"{config['model_name']} | {config['feature_setting']}"] = classification_report(
        y_test, predictions, zero_division=0
    )

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df = baseline_results_df.sort_values(
    by=['Weighted F1', 'Accuracy'], ascending=False
).reset_index(drop=True)

display(baseline_results_df)

# Save the results table for manuscript reporting
baseline_results_df.to_csv('supervised_baseline_results.csv', index=False)
print('Saved supervised baseline results to supervised_baseline_results.csv')


Training Multinomial Naive Bayes | TF-IDF unigram + bigram...
Training Logistic Regression | TF-IDF unigram + bigram...
Training Random Forest | TF-IDF unigram + bigram...
Training Linear SVM | TF-IDF unigram...
Training Linear SVM | TF-IDF unigram + bigram [proposed]...
Training Linear SVM | TF-IDF unigram + bigram + trigram...


,Model,Feature Setting,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Linear SVM,TF-IDF unigram,0.9420,0.9462,0.9420,0.9431,0.9431
1,Linear SVM,TF-IDF unigram + bigram [proposed],0.9358,0.9391,0.9358,0.9368,0.9368
2,Linear SVM,TF-IDF unigram + bigram + trigram,0.9323,0.9358,0.9323,0.9334,0.9334
3,Logistic Regression,TF-IDF unigram + bigram,0.9268,0.9399,0.9268,0.9300,0.9300
4,Random Forest,TF-IDF unigram + bigram,0.9178,0.9353,0.9178,0.9221,0.9221
5,Multinomial Naive Bayes,TF-IDF unigram + bigram,0.8923,0.9334,0.8923,0.9027,0.9027


Saved supervised baseline results to supervised_baseline_results.csv


In [19]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder

# Select text column safely
text_col = "text" if "text" in df.columns else "content"

X_text = df[text_col].fillna("").astype(str)
y_true = df["category"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_true)

# ------------------------------------------------------------
# LDA evaluation
# ------------------------------------------------------------

count_vectorizer = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 1),
    min_df=1,
    max_df=1.0
)

X_counts = count_vectorizer.fit_transform(X_text)

lda = LatentDirichletAllocation(
    n_components=8,
    random_state=42,
    learning_method="batch"
)

lda_doc_topic = lda.fit_transform(X_counts)
lda_pred = lda_doc_topic.argmax(axis=1)

lda_perplexity = lda.perplexity(X_counts)
lda_log_likelihood = lda.score(X_counts)

lda_silhouette = silhouette_score(lda_doc_topic, lda_pred)
lda_ari = adjusted_rand_score(y_encoded, lda_pred)
lda_nmi = normalized_mutual_info_score(y_encoded, lda_pred)

lda_feature_names = np.array(count_vectorizer.get_feature_names_out())

lda_topics = []
for topic_idx, topic in enumerate(lda.components_):
    top_words = lda_feature_names[topic.argsort()[-10:][::-1]]
    lda_topics.append({
        "Model": "LDA",
        "Topic": topic_idx + 1,
        "Top Words": ", ".join(top_words)
    })

lda_topics_df = pd.DataFrame(lda_topics)

# ------------------------------------------------------------
# LSI evaluation
# ------------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 1),
    min_df=1,
    max_df=1.0,
    norm="l2"
)

X_tfidf = tfidf_vectorizer.fit_transform(X_text)

lsi = TruncatedSVD(
    n_components=8,
    random_state=42
)

lsi_doc_topic = lsi.fit_transform(X_tfidf)

kmeans = KMeans(
    n_clusters=8,
    random_state=42,
    n_init=10
)

lsi_pred = kmeans.fit_predict(lsi_doc_topic)

lsi_silhouette = silhouette_score(lsi_doc_topic, lsi_pred)
lsi_ari = adjusted_rand_score(y_encoded, lsi_pred)
lsi_nmi = normalized_mutual_info_score(y_encoded, lsi_pred)
lsi_explained_variance = lsi.explained_variance_ratio_.sum()

lsi_feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

lsi_topics = []
for topic_idx, component in enumerate(lsi.components_):
    top_words = lsi_feature_names[component.argsort()[-10:][::-1]]
    lsi_topics.append({
        "Model": "LSI",
        "Topic": topic_idx + 1,
        "Top Words": ", ".join(top_words)
    })

lsi_topics_df = pd.DataFrame(lsi_topics)

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

unsupervised_results_df = pd.DataFrame([
    {
        "Model": "LDA",
        "Topics/Clusters": 8,
        "Perplexity": round(lda_perplexity, 4),
        "Log Likelihood": round(lda_log_likelihood, 4),
        "Explained Variance": "N/A",
        "Silhouette Score": round(lda_silhouette, 4),
        "Adjusted Rand Index": round(lda_ari, 4),
        "Normalized Mutual Information": round(lda_nmi, 4)
    },
    {
        "Model": "LSI + KMeans",
        "Topics/Clusters": 8,
        "Perplexity": "N/A",
        "Log Likelihood": "N/A",
        "Explained Variance": round(lsi_explained_variance, 4),
        "Silhouette Score": round(lsi_silhouette, 4),
        "Adjusted Rand Index": round(lsi_ari, 4),
        "Normalized Mutual Information": round(lsi_nmi, 4)
    }
])

print("Unsupervised model evaluation:")
display(unsupervised_results_df)

print("LDA topic examples:")
display(lda_topics_df)

print("LSI topic examples:")
display(lsi_topics_df)

unsupervised_results_df.to_csv("unsupervised_baseline_results.csv", index=False)
lda_topics_df.to_csv("lda and lsa.csv", index=False)
lsi_topics_df.to_csv("lda and lsa.csv", index=False)

Unsupervised model evaluation:


,Model,Topics/Clusters,Perplexity,Log Likelihood,Explained Variance,Silhouette Score,Adjusted Rand Index,Normalized Mutual Information
0,LDA,8,569.0081,-5267583.117,N/A,0.6891,0.1828,0.3508
1,LSI + KMeans,8,N/A,N/A,0.1979,0.6032,0.5283,0.7540


LDA topic examples:


,Model,Topic,Top Words
0,LDA,1,"ay, ee, oo, ka, ku, in, ayaa, ah, uu, iyo"
1,LDA,2,"oo, ka, ku, ay, ah, uu, in, ee, ayaa, la"
2,LDA,3,"iyo, ee, soomaaliya, oo, ayaa, ka, ku, ah, dal..."
3,LDA,4,"oo, ka, ayaa, uu, khan, lagu, in, internet, sa..."
4,LDA,5,"oo, ayaa, ah, ka, cusub, lagu, la, ee, soomaal..."
5,LDA,6,"oo, cusub, ka, ayaa, ah, soomaaliyeed, ee, soo..."
6,LDA,7,"oo, ay, ah, ka, in, ku, ee, la, iyo, ayaa"
7,LDA,8,"oo, ka, ayaa, ah, ay, ku, iyo, in, ee, lagu"


LSI topic examples:


,Model,Topic,Top Words
0,LSI,1,"oo, ka, ah, ay, ee, ku, ayaa, cusub, la, in"
1,LSI,2,"cusub, bandhigga, app, soomaaliyeed, tiknoolaj..."
2,LSI,3,"diimeed, culimada, muxaadaro, masjidka, isbaha..."
3,LSI,4,"bilaabay, bangiyada, adeegyo, ah, ganacsi, hes..."
4,LSI,5,"koox, xulka, qabtay, gobollada, kubadda, ciyaa..."
5,LSI,6,"daawaday, soomaaliyeed, wuxuu, bandhig, hadlay..."
6,LSI,7,"caafimaadka, cusub, loo, caafimaad, soomaaliye..."
7,LSI,8,"muqdisho, caafimaadka, gobollada, yimid, karii..."


In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy import stats

# Select text column safely
text_col = "text" if "text" in df.columns else "content"

X = df[text_col].fillna("").astype(str)
y = df["category"]

# Final proposed model: Linear SVM with TF-IDF unigram
proposed_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 1))),
    ("clf", LinearSVC(C=1.0))
])

# ------------------------------------------------------------
# 1. Repeated stratified 80/20 holdout evaluation
# ------------------------------------------------------------

seeds = [0, 1, 2, 3, 4, 5, 10, 21, 42, 100]
repeated_results = []

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    proposed_model.fit(X_train, y_train)
    y_pred = proposed_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )

    repeated_results.append({
        "seed": seed,
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

repeated_results_df = pd.DataFrame(repeated_results)

# Summary with mean, standard deviation, and 95% confidence interval
summary_rows = []

for metric in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]:
    values = repeated_results_df[metric].values
    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = stats.t.interval(
        confidence=0.95,
        df=len(values) - 1,
        loc=mean,
        scale=std / np.sqrt(len(values))
    )

    summary_rows.append({
        "Metric": metric,
        "Mean": round(mean, 4),
        "Std. Dev.": round(std, 4),
        "95% CI Lower": round(ci_low, 4),
        "95% CI Upper": round(ci_high, 4)
    })

repeated_summary_df = pd.DataFrame(summary_rows)

print("Repeated stratified holdout results:")
display(repeated_results_df)

print("Repeated-run summary:")
display(repeated_summary_df)

repeated_results_df.to_csv("repeated_holdout_results.csv", index=False)
repeated_summary_df.to_csv("repeated_holdout_summary.csv", index=False)


# ------------------------------------------------------------
# 2. Stratified 5-fold cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "macro_precision": "precision_macro",
    "macro_recall": "recall_macro",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted"
}

cv_results = cross_validate(
    proposed_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

cv_summary_rows = []

for metric in scoring.keys():
    values = cv_results[f"test_{metric}"]
    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = stats.t.interval(
        confidence=0.95,
        df=len(values) - 1,
        loc=mean,
        scale=std / np.sqrt(len(values))
    )

    cv_summary_rows.append({
        "Metric": metric,
        "Mean": round(mean, 4),
        "Std. Dev.": round(std, 4),
        "95% CI Lower": round(ci_low, 4),
        "95% CI Upper": round(ci_high, 4)
    })

cv_summary_df = pd.DataFrame(cv_summary_rows)

print("5-fold cross-validation summary:")
display(cv_summary_df)

cv_summary_df.to_csv("cross_validation_summary.csv", index=False)

Repeated stratified holdout results:


,seed,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,0,0.926796,0.932570,0.926796,0.928542,0.928542
1,1,0.918508,0.926282,0.918508,0.920402,0.920402
2,2,0.928177,0.935076,0.928177,0.930004,0.930004
3,3,0.933011,0.937000,0.933011,0.933894,0.933894
4,4,0.930939,0.933867,0.930939,0.931866,0.931866
5,5,0.935083,0.939378,0.935083,0.936476,0.936476
6,10,0.941298,0.943725,0.941298,0.941957,0.941957
7,21,0.938536,0.942956,0.938536,0.939643,0.939643
8,42,0.939227,0.943010,0.939227,0.940288,0.940288
9,100,0.919890,0.926160,0.919890,0.921829,0.921829


Repeated-run summary:


,Metric,Mean,Std. Dev.,95% CI Lower,95% CI Upper
0,accuracy,0.9311,0.0079,0.9255,0.9368
1,macro_precision,0.9360,0.0065,0.9314,0.9406
2,macro_recall,0.9311,0.0079,0.9255,0.9368
3,macro_f1,0.9325,0.0075,0.9272,0.9378
4,weighted_f1,0.9325,0.0075,0.9272,0.9378


5-fold cross-validation summary:


,Metric,Mean,Std. Dev.,95% CI Lower,95% CI Upper
0,accuracy,0.9320,0.0097,0.9200,0.9441
1,macro_precision,0.9367,0.0076,0.9273,0.9461
2,macro_recall,0.9320,0.0097,0.9200,0.9441
3,macro_f1,0.9333,0.0091,0.9220,0.9447
4,weighted_f1,0.9333,0.0091,0.9220,0.9447


In [20]:
import time
import os
import tempfile
import joblib
import platform
import sklearn
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# Select text column safely
text_col = "text" if "text" in df.columns else "content"

X = df[text_col].fillna("").astype(str)
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 1),
        min_df=1,
        max_df=1.0,
        norm="l2",
        use_idf=True,
        smooth_idf=True
    )),
    ("clf", LinearSVC(C=1.0))
])

# Training time
start_train = time.perf_counter()
model.fit(X_train, y_train)
end_train = time.perf_counter()

training_time = end_train - start_train

# Inference time
start_pred = time.perf_counter()
y_pred = model.predict(X_test)
end_pred = time.perf_counter()

inference_time = end_pred - start_pred
inference_time_per_article_ms = (inference_time / len(X_test)) * 1000

# Performance
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

# Model size
tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".joblib")
joblib.dump(model, tmp.name)
model_size_mb = os.path.getsize(tmp.name) / (1024 * 1024)
os.remove(tmp.name)

resource_results = pd.DataFrame([{
    "Model": "TF-IDF unigram + Linear SVM",
    "Training Time (seconds)": round(training_time, 4),
    "Total Inference Time (seconds)": round(inference_time, 4),
    "Inference Time per Article (ms)": round(inference_time_per_article_ms, 4),
    "Serialized Model Size (MB)": round(model_size_mb, 4),
    "Accuracy": round(accuracy, 4),
    "Macro F1": round(macro_f1, 4),
    "Weighted F1": round(weighted_f1, 4),
    "Hardware/Processor": platform.processor(),
    "Python Version": platform.python_version(),
    "scikit-learn Version": sklearn.__version__
}])

display(resource_results)
resource_results.to_csv("resource_efficiency_results.csv", index=False)

,Model,Training Time (seconds),Total Inference Time (seconds),Inference Time per Article (ms),Serialized Model Size (MB),Accuracy,Macro F1,Weighted F1,Hardware/Processor,Python Version,scikit-learn Version
0,TF-IDF unigram + Linear SVM,1.7469,0.2135,0.1474,0.4835,0.9392,0.9403,0.9403,x86_64,3.12.13,1.6.1


In [21]:
from sklearn.metrics import classification_report

# Exact overlap check between train and test
train_texts = set(X_train)
test_texts = set(X_test)
overlap_count = len(train_texts.intersection(test_texts))

print("Exact train-test text overlap:", overlap_count)

# Optional: source-based split if source column exists
if "source" in df.columns:
    print(df["source"].value_counts())

    sources = df["source"].dropna().unique()
    print("Available sources:", sources)

    # Example: train on BBC, test on VOA
    bbc_mask = df["source"].str.lower().str.contains("bbc", na=False)
    voa_mask = df["source"].str.lower().str.contains("voa|voice", na=False)

    if bbc_mask.sum() > 0 and voa_mask.sum() > 0:
        X_train_source = df.loc[bbc_mask, text_col].fillna("").astype(str)
        y_train_source = df.loc[bbc_mask, "category"]
        X_test_source = df.loc[voa_mask, text_col].fillna("").astype(str)
        y_test_source = df.loc[voa_mask, "category"]

        source_model = model
        source_model.fit(X_train_source, y_train_source)
        y_pred_source = source_model.predict(X_test_source)

        print("Train on BBC, test on VOA")
        print(classification_report(y_test_source, y_pred_source, digits=4))

# Optional: time-based split if date column exists
if "date" in df.columns:
    df_time = df.copy()
    df_time["date"] = pd.to_datetime(df_time["date"], errors="coerce")
    df_time = df_time.dropna(subset=["date"]).sort_values("date")

    split_idx = int(len(df_time) * 0.8)

    train_time = df_time.iloc[:split_idx]
    test_time = df_time.iloc[split_idx:]

    X_train_time = train_time[text_col].fillna("").astype(str)
    y_train_time = train_time["category"]
    X_test_time = test_time[text_col].fillna("").astype(str)
    y_test_time = test_time["category"]

    time_model = model
    time_model.fit(X_train_time, y_train_time)
    y_pred_time = time_model.predict(X_test_time)

    print("Time-based split results")
    print(classification_report(y_test_time, y_pred_time, digits=4))

Exact train-test text overlap: 175


In [22]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

text_col = "text" if "text" in df.columns else "content"

def normalize_text_for_deduplication(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df_clean = df.copy()
df_clean["dedup_key"] = df_clean[text_col].apply(normalize_text_for_deduplication)

# Check duplicates before removal
duplicate_rows = df_clean.duplicated(subset=["dedup_key"]).sum()
print("Duplicate rows before removal:", duplicate_rows)

# Check whether identical texts have conflicting labels
label_conflicts = (
    df_clean.groupby("dedup_key")["category"]
    .nunique()
    .reset_index()
)

conflicting_duplicates = label_conflicts[label_conflicts["category"] > 1]
print("Identical texts with conflicting labels:", len(conflicting_duplicates))

# Remove exact duplicate article texts before splitting
df_dedup = df_clean.drop_duplicates(subset=["dedup_key"], keep="first").reset_index(drop=True)

print("Original dataset size:", len(df))
print("Deduplicated dataset size:", len(df_dedup))
print("Removed duplicate rows:", len(df) - len(df_dedup))

print("\nClass distribution after deduplication:")
print(df_dedup["category"].value_counts())

X = df_dedup[text_col].fillna("").astype(str)
y = df_dedup["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Confirm no exact train-test overlap
train_texts = set(X_train)
test_texts = set(X_test)
overlap_count = len(train_texts.intersection(test_texts))
print("\nExact train-test text overlap after deduplication:", overlap_count)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 1),
        min_df=1,
        max_df=1.0,
        norm="l2",
        use_idf=True,
        smooth_idf=True
    )),
    ("clf", LinearSVC(C=1.0))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 4))
print("Weighted F1:", round(f1_score(y_test, y_pred, average="weighted"), 4))

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

Duplicate rows before removal: 4657
Identical texts with conflicting labels: 0
Original dataset size: 7240
Deduplicated dataset size: 2583
Removed duplicate rows: 4657

Class distribution after deduplication:
category
Siyaasad         903
Dagaal           704
Caafimaad        292
Ciyaaro          176
Ganacsi          167
Tiknoolajiyad    133
Madadaalo        122
Diini             86
Name: count, dtype: int64

Exact train-test text overlap after deduplication: 0

Accuracy: 0.8182
Macro F1: 0.73
Weighted F1: 0.8067

Classification report:
               precision    recall  f1-score   support

    Caafimaad     0.7188    0.7797    0.7480        59
      Ciyaaro     0.8800    0.6286    0.7333        35
       Dagaal     0.9097    0.9291    0.9193       141
        Diini     0.8000    0.7059    0.7500        17
      Ganacsi     0.6429    0.2727    0.3830        33
    Madadaalo     0.9412    0.6667    0.7805        24
     Siyaasad     0.7880    0.9448    0.8593       181
Tiknoolajiyad   

In [18]:
def predict_from_title_content(title, content):
    combined = f"{title} {content}"
    text_cleaner = TextCleaner()
    cleaned_text = text_cleaner.clean_text(combined)
    return pipeline.predict([combined])[0]

# Usage
title = "Trump oo sheegay inuu wici doono Putin, muxuuse kala hadlayaa"
content = "Madaxweynaha Marayanka Donald Trump ayaa sheegay inuu la hadli doono madaxweynaha Ruushka Vladimir Putin isniinta si ay ugu wadahadlaan joojinta colaada Ukraine, isagoo sheegay in wicitaanku ku saabsanaan doono joojinta daadinta dhiigga. Qoraal lagu daabacay barta bulshada ee Truth Social, Madaxweynaha Mareykanka ayaa sheegay in wicitaanka uu dhici doono 10:00 subaxnimo waqtiga Mareykanka (14:00 GMT), kadibna uu la hadli doono Madaxweynaha Ukraine Volodymyr Zelensky iyo qaar ka mid ah madaxda waddamada gaashaanbuurta NATO. Ruushka iyo Ukraine kama aysan gaarin wax horumar weyn ah kulankii fool ka foolka ahaa ee ugu horreeyay muddo saddex sano ah oo ka dhacay magaalada Istanbul Jimcihii, inkasta oo ay isla oggolaadeen is dhaafsiga maxaabiista. Trump wuxuu soo jeediyay inuu ka qayb galo wadahadalladii ka dhacay Turkeyga haddii Putin uu isna tagayo, balse Madaxweynaha Ruushka ayaa diiday inuu yimaado wadahadalladii ka dhacay dalka Turkeyga."
print(predict_from_title_content(title, content))


Siyaasad


In [ ]:
import joblib

# Save pipeline to file
joblib.dump(pipeline, 'somali_news_classifier_model.joblib')
print("Model saved successfully!")


In [ ]:
# Load pipeline from file
loaded_pipeline = joblib.load('somali_news_classifier_model.joblib')
print("Model loaded successfully!")


In [ ]:
new_title = "Tukaankeyga oo aan si fiican u camiray ayaan habeenkii ka tagay, subixiina waxa aan u imid dambas kaliya"
new_content = "Khasaare kala duwan ayaa ka dhashay dab ka kacay suuqa bakaaraha ee magaalada Muqdisho. Dabkan oo ka billowday qaybta bagaashka lagu iibiyo ee u dhexeysa qaybta loo yaqaano Dahablaha iyo Khaliifa ayaa si dakhso leh ugu faafay dhul baaxad leh, halkaas oo ay ku hanti beeleen dad u badan ganacsatada dharka iyo kabaha iibiya. Illa hadda lama shaacin sababta ka dambeysey dabkan oo ah kii labaad oo ka kaca suuqa ugu weyn dalka muddo bil gudaheed ah, hase yeeshee ganacsatada qaar ayaa sheegay in dabku ay koronto dhalisay. Tobanaan ganacsato ah ayaa ku hanti belay dabka iyadoo ay ahayd xilli uu suuqu aad u mashquul badanyahay loona diyaar garoobayay munaasabadda ciidul fidriga oo maalmo qura ay ka dhimanaayeen."

combined_text = new_title + " " + new_content

predicted_label = loaded_pipeline.predict([combined_text])[0]
print(f"Predicted category: {predicted_label}")